### Install Dependencies

In [0]:
!pip install databricks-sdk -U
dbutils.library.restartPython()

### Define Parameters

Review the widgets before running. Defaults match the **Marketing Campaign Analytics** layout.

| Widget | Default | What it controls |
| --- | --- | --- |
| `GENIE_SPACE_ID` | *(empty)* | Optional. If blank, the space is auto-found from `GENIE_SPACE_TITLE` — no copy-paste needed. |
| `GENIE_SPACE_TITLE` | `Marketing Campaign Analytics — Genie Space` | Title of the Genie space created in `2_genie/0`. Used to auto-resolve `GENIE_SPACE_ID`. |
| `QUESTION` | `What are my top 5 campaigns by total sales?` | Natural-language question sent to Genie. |


In [0]:
dbutils.widgets.text("GENIE_SPACE_ID", "", "GENIE_SPACE_ID — optional; auto-resolved from GENIE_SPACE_TITLE if left blank")
dbutils.widgets.text("GENIE_SPACE_TITLE", "Marketing Campaign Analytics — Genie Space", "GENIE_SPACE_TITLE — title used in 2_genie/0 (used to auto-find the space)")
dbutils.widgets.text("QUESTION", "What are my top 5 campaigns by total sales?", "QUESTION — natural-language question for Genie")


### Read Parameters

The Genie space id is auto-resolved from `GENIE_SPACE_TITLE` if you didn't paste one.

In [ ]:
GENIE_SPACE_ID = dbutils.widgets.get("GENIE_SPACE_ID").strip()
GENIE_SPACE_TITLE = dbutils.widgets.get("GENIE_SPACE_TITLE").strip()
QUESTION = dbutils.widgets.get("QUESTION")

if not GENIE_SPACE_ID:
    from databricks.sdk import WorkspaceClient
    _w = WorkspaceClient()
    _spaces, _token = [], None
    while True:
        _resp = _w.genie.list_spaces(page_size=100, page_token=_token)
        _spaces.extend(list(getattr(_resp, "spaces", []) or []))
        _token = getattr(_resp, "next_page_token", None)
        if not _token:
            break
    _match = next((s for s in _spaces if (getattr(s, "title", "") or "") == GENIE_SPACE_TITLE), None)
    assert _match is not None, (
        f"No Genie space found with title '{GENIE_SPACE_TITLE}'. "
        "Run 2_genie/0_CreateGenieSpace first, or paste the space id into the GENIE_SPACE_ID widget."
    )
    GENIE_SPACE_ID = _match.space_id
    print(f"Auto-resolved Genie space '{GENIE_SPACE_TITLE}' -> {GENIE_SPACE_ID}")

print("Genie space :", GENIE_SPACE_ID)
print("Question    :", QUESTION)

### Query Genie Space via API

Uses the [Genie Conversation API](https://docs.databricks.com/api/workspace/genie) to ask a natural-language question and poll for the answer.

In [0]:
import json, time
import requests
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
host = w.config.host
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

# Start conversation
resp = requests.post(
    f"{host}/api/2.0/genie/spaces/{GENIE_SPACE_ID}/start-conversation",
    headers=headers,
    json={"content": QUESTION},
)
resp.raise_for_status()
msg = resp.json()
conversation_id = msg["conversation_id"]
message_id = msg["message_id"]
print(f"Conversation: {conversation_id}")

# Poll until complete
for _ in range(60):
    poll = requests.get(
        f"{host}/api/2.0/genie/spaces/{GENIE_SPACE_ID}/conversations/{conversation_id}/messages/{message_id}",
        headers=headers,
    )
    poll.raise_for_status()
    data = poll.json()
    status = data.get("status")
    if status in ("COMPLETED", "FAILED", "CANCELLED"):
        break
    time.sleep(2)

print("Status:", data.get("status"))
for att in data.get("attachments", []):
    if att.get("text"):
        print("\nAnswer:\n", att["text"][:2000])
